In [9]:
import pandas as pd
import numpy as np
import joblib
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the 82k emails
# Make sure your 'emails.csv' is inside the 'data' folder
df = pd.read_csv(r'C:\Users\vikas\OneDrive\Documents\Phishing-Email-Detection-System\data\emails.csv')

print(f"Total Emails Loaded: {len(df)}")
print(df['label'].value_counts()) # Shows how many are Phishing (1) vs Safe (0)

Total Emails Loaded: 82486
label
1    42891
0    39595
Name: count, dtype: int64


In [10]:
def clean_text(text):
    text = str(text).lower()
    # This keeps only letters, numbers, and spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text

df['text'] = df['text'].apply(clean_text)
print("Cleaning complete. Sample text:")
print(df['text'].iloc[0][:100])

Cleaning complete. Sample text:
hpl nom may 25 2001 see attached file hplno 525 xls hplno 525 xls


In [11]:
MAX_WORDS = 15000
MAX_LEN = 300

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

# IMPORTANT: Save this to your backend folder immediately
os.makedirs('backend/models', exist_ok=True)
joblib.dump(tokenizer, 'backend/models/tokenizer.pkl')

# Convert text to sequences of numbers
X = tokenizer.texts_to_sequences(df['text'])
X = pad_sequences(X, maxlen=MAX_LEN)
y = df['label'].values

print("Dictionary created and saved to 'backend/models/tokenizer.pkl'")

Dictionary created and saved to 'backend/models/tokenizer.pkl'


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Start Training
print("Starting Training (this might take a few minutes)...")
model.fit(X, y, epochs=5, batch_size=64, validation_split=0.2)

# Save the finished Brain
model.save('backend/models/cnn_phishing_model.h5')
print("✅ Brain saved to 'backend/models/cnn_phishing_model.h5'")

c:\Users\vikas\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Starting Training (this might take a few minutes)...
Epoch 1/5
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 97s 84ms/step - accuracy: 0.9729 - loss: 0.0714 - val_accuracy: 0.8095 - val_loss: 0.7620
Epoch 2/5
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 99s 96ms/step - accuracy: 0.9967 - loss: 0.0111 - val_accuracy: 0.8101 - val_loss: 1.4016
Epoch 3/5
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 86s 83ms/step - accuracy: 0.9990 - loss: 0.0037 - val_accuracy: 0.7957 - val_loss: 1.1523
Epoch 4/5
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 85s 82ms/step - accuracy: 0.9995 - loss: 0.0019 - val_accuracy: 0.8074 - val_loss: 1.7950
Epoch 5/5
1032/1032 ━━━━━━━━━━━━━━━━━━━━ 87s 84ms/step - accuracy: 0.9997 - loss: 0.0014 - val_accuracy: 0.8076 - val_loss: 1.6439


✅ Brain saved to 'backend/models/cnn_phishing_model.h5'


In [15]:
from sklearn.model_selection import train_test_split

# Splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_test is now defined. Shape: {X_test.shape}")

X_test is now defined. Shape: (16498, 300)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Ensure the data is split (Using the X and y from your earlier cells)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Get predictions from your trained model
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# 3. Calculate metrics for TABLE I
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-Score": f1_score(y_test, y_pred)
}

print("\n--- TABLE I: PERFORMANCE METRICS ---")
for k, v in metrics.items():
    print(f"{k}: {v*100:.2f}%")

# 4. Generate FIG 4: CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Safe', 'Phishing'], 
            yticklabels=['Safe', 'Phishing'])
plt.title('Fig 4: Confusion Matrix - Phishing Shield PRO')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()